<h1 style="color:#6291be;font-weight:600;">Capture Patterns with Recurrent Neural Networks</h1>

The previous models we created, removed word ordering, and thus cannot be used to solve more complex or ambiguoys tasks such as **text generation** or **question answering**

> **Why use a RNN?**<br/>
> To capture the meaning of a a text sequence.
>
> **How it works?**<br/>
> We pass our sentence through the network, one token at a time. The network produces some **state** which is passed to the network again together with the next token.

![RNN](./img/image.png)

$X_0, ..., X_n$ is the input sequence of tokens (words?).<br/>
The RNN creates a corresponding sequence of neural network cells. All cells share the same weights and are trained using one backpropagation pass.<br/>
Each cell block takes as input a pair $(X_i, S_i)$ and outputs the next state $S_{i+1}$.<br/>
The final state goes through a linear classifier (the Dense layer) to produce the final result.

Each RNN cell contains two weight matrices $W_H$ and $W_I$ and bias b. Given an input token $X_i$ and an input state $S_i$, the output state is calculated as:

$S_{i+1} = f(W_H \times S_i + W_I \times X_i + b)$

f is an **activation function**, usually tanh.

In [7]:
import os

# Set the backend that keras uses before importing keras.
os.environ["KERAS_BACKEND"] = "tensorflow"

# Suppress warnings about GPU support.
# "2" hides INFO and WARNING messages but still shows ERROR messages
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # 0=ALL, 1=INFO, 2=WARNING, 3=ERROR

import tensorflow as tf
import keras
import tensorflow_datasets as tfds
import numpy as np

In [8]:
dataset = tfds.load("ag_news_subset")
ds_train = dataset["train"]
ds_test = dataset["test"]

batch_size = 16
embed_size = 64

<h3 style="font-weight:600;color:white;background:#6291be;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">1 | Simple RNN Classifier</h3>

In [11]:
# We use a smaller vocabulary (20,000) here than in previous units because
# RNN models have more parameters per token, and a smaller vocabulary
# helps keep training time and memory usage manageable.
vocab_size = 20_000
vectorizer = keras.layers.TextVectorization(max_tokens=vocab_size)

model = keras.Sequential(
    [
        keras.Input(shape=(1,), dtype=tf.string),
        vectorizer,
        keras.layers.Embedding(vocab_size, embed_size),
        keras.layers.SimpleRNN(16),
        keras.layers.Dense(4, activation="softmax"),
    ]
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_2            │ (None, None)           │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, None, 64)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 16)             │         1,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,281,364 (4.89 MB)

 Trainable params: 1,281,364 (4.89 MB)

 Non-trainable params: 0 (0.00 B)

<span style="color:#6291be;font-weight:600;text-transform:uppercase;">Train our RNN</span>

We'll only train the RNN model on news titles, to speed up things, omitting the description.

In [12]:
def extract_title(x):
    return x["title"]


def tupelize_title(x):
    return (extract_title(x), x["label"])


print("Training vectorizer")
vectorizer.adapt(ds_train.take(2000).map(extract_title))

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

model.fit(
    ds_train.map(tupelize_title).batch(batch_size),
    validation_data=ds_test.map(tupelize_title).batch(batch_size),
)

Training vectorizer
7500/7500 ━━━━━━━━━━━━━━━━━━━━ 62s 8ms/step - accuracy: 0.7688 - loss: 0.6346 - val_accuracy: 0.7825 - val_loss: 0.5962


<h3 style="font-weight:600;color:white;background:#6291be;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">2 | Revisiting Variable Sequences</h3>

`TextVectorization`<br/>
When we have sequences of variable length, this layer automatically pads them with pad tokens. But these pads can complicate covergence of the model.

`Embedding`<br/>
We can use masking in this layer to minimize the amount of padding. To achieve this specify **mask_zero=True** parameter. In this case token index 0, isn't a valid vocab entry, and so all entries are shifted by 1.

So now let's train the model on the whole dataset of *both* titles and descriptions:

In [13]:
model = keras.Sequential(
    [
        keras.Input(shape=(1,), dtype=tf.string),
        vectorizer,
        keras.layers.Embedding(vocab_size, embed_size, mask_zero=True),
        keras.layers.SimpleRNN(16),
        keras.layers.Dense(4, activation="softmax"),
    ]
)


def extract_text(x):
    return x["title"] + " " + x["description"]


def tupelize(x):
    return (extract_text(x), x["label"])


print("Training vectorizer")
vectorizer.adapt(ds_train.take(2000).map(extract_text))

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

model.fit(
    ds_train.map(tupelize).batch(batch_size),
    validation_data=ds_test.map(tupelize).batch(batch_size),
)

Training vectorizer
7500/7500 ━━━━━━━━━━━━━━━━━━━━ 128s 17ms/step - accuracy: 0.8311 - loss: 0.4900 - val_accuracy: 0.8925 - val_loss: 0.3523


<h3 style="font-weight:600;color:white;background:#6291be;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">3 | LSTM: Long short-term memory</h3>

> **The problem of vanishing gradients**<br/>
> RNNs can be long, making it difficult to propagate the gradients all the way back to the first layer of the network during backpropagation.
>
> **SOLUTION: explicit state management by using gates**<br/>
> We can introduce gates using **long short-term memory**.

![LSTM](./img/image_02.png)

LSTMs have the following gates (from left to right on the picture above):

- **forget gate** which determines which components of the vector we need to forget, and which to pass through.
- **input gate** which determines how much information from the input vector and previous hidden vector should be incorporated into the state vector.
- **output gate** which takes the new state vector and decides which of its components will be used to produce the new hidden vector 
.

In [14]:
model = keras.Sequential(
    [
        keras.Input(shape=(1,), dtype=tf.string),
        vectorizer,
        keras.layers.Embedding(vocab_size, embed_size),
        keras.layers.LSTM(8),
        keras.layers.Dense(4, activation="softmax"),
    ]
)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

model.fit(
    ds_train.map(tupelize).batch(batch_size),
    validation_data=ds_test.map(tupelize).batch(batch_size),
)

7500/7500 ━━━━━━━━━━━━━━━━━━━━ 125s 16ms/step - accuracy: 0.6755 - loss: 0.7159 - val_accuracy: 0.8411 - val_loss: 0.5120


<span style="text-decoration:underline;color:#6291be;font-weight:600;font-style:italic;">Further Reading:</span>

[Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)<br/>
[The Unreasonable Effectiveness of Recurrent Neural Networks](https://karpathy.github.io/2015/05/21/rnn-effectiveness/)

<h3 style="font-weight:600;color:white;background:#6291be;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">4 | Bidirectional and Multilayer RNNs</h3>

For scenarios that require random access of the input sequence, it makes more sense to run the recurrent computation in both directions. This is achieved by wrapping the recurrent layer with a `Bidirectional` layer.

For all layers, except the last one, we set `return_sequences=True`, because we need the layer to return all intermediate states, and not just the final state of the recurrent computation.

In [16]:
model = keras.Sequential(
    [
        keras.Input(shape=(1,), dtype=tf.string),
        vectorizer,
        keras.layers.Embedding(vocab_size, 128, mask_zero=True),
        keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True)),
        keras.layers.Bidirectional(keras.layers.LSTM(64)),
        keras.layers.Dense(4, activation="softmax"),
    ]
)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

model.fit(
    ds_train.map(tupelize).batch(batch_size),
    validation_data=ds_test.map(tupelize).batch(batch_size),
)

7500/7500 ━━━━━━━━━━━━━━━━━━━━ 586s 78ms/step - accuracy: 0.8927 - loss: 0.3166 - val_accuracy: 0.9096 - val_loss: 0.2717
